# 03 · 完整实验矩阵与断点恢复

默认只创建 manifest，不运行模型。完整矩阵由 `configs/full_experiments.json` 的 jobs 决定；每个 job 指向冻结配置。所有论文结果来自真实执行的 `result.json`，不能填入引用论文数值。

只有 G3 PASS、证据哈希匹配、实验后端已实现及当前配置未改变时才可启动。G1/G2 单独通过或数学 toy 通过均不足以启动本册。

预算是 GPU 小时上限，不是性能承诺。停止/重启时只跳过签名及结果哈希完全匹配的已完成作业；失败作业不跳过。

执行器会用当前 `gate_rules.json` 重新计算 G3，核对原报告规则哈希，并核对 `provenance.execution_fingerprint_sha256` 与当前执行指纹。手工填写 PASS 或沿用旧 GPU 的通过报告都不能解锁执行。

## 路径与环境配置

CPU 默认路径不下载模型、不调用付费服务、不自动升级 PyTorch。**只上传 ipynb 不够，必须一起提供 src/configs/docs。** 首次在 Colab 上传整个 ZIP（或放 Drive），设置 `KIT_ARCHIVE`，将 `EXTRACT_ARCHIVE=True`，即可用下面单元安全解压；Drive 路径先设置 `MOUNT_DRIVE=True`。如果已经手动解压，则直接设置 `PROJECT_ROOT`。缺少基础 Python 依赖时才把 `INSTALL_BASE_DEPENDENCIES=True`。Colab 断线会丢失 `/content`，需要在阶段完成后同步结果；高频权重与激活 I/O 应留在本地磁盘。

访问受限 LLaMA 权重需要你已有的 Hugging Face 访问权。凭据只通过环境变量或 Colab Secrets 提供，不能写入 notebook、配置、提示词、日志或 Git。

In [1]:
from pathlib import Path
import os, sys, json

# Colab: 解压整个项目到 /content/cellquant_research_kit；不要仅上传 ipynb。
# AutoDL: 建议 /root/autodl-tmp/cellquant_research_kit。
# 留 None 时依次查找当前目录、父目录和上述两个位置。
PROJECT_ROOT = os.environ.get("CELLQUANT_PROJECT_ROOT") or None
RUN_ROOT = os.environ.get("CELLQUANT_RUN_ROOT") or None
KIT_ARCHIVE = None  # 例：/content/drive/MyDrive/cellquant_research_kit.zip
EXTRACT_ARCHIVE = False  # 首次运行且 ZIP 已上传时才改 True
ARCHIVE_DESTINATION = "/content/cellquant_workspace" if Path("/content").exists() else str(Path.cwd().parent / "cellquant_workspace")
MOUNT_DRIVE = False
DURABLE_ROOT = None  # 例：/content/drive/MyDrive/cellquant_runs；不存权重或 token
INSTALL_BASE_DEPENDENCIES = False
SEED = 20261115

In [2]:
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Drive mounting skipped; outputs remain under RUN_ROOT until copied.")

Drive mounting skipped; outputs remain under RUN_ROOT until copied.


In [3]:
archive_roots = []
if EXTRACT_ARCHIVE:
    import zipfile, hashlib
    if not KIT_ARCHIVE or not Path(KIT_ARCHIVE).is_file():
        raise FileNotFoundError("先上传完整 ZIP 并设置 KIT_ARCHIVE")
    archive_path = Path(KIT_ARCHIVE)
    destination = Path(ARCHIVE_DESTINATION).resolve()
    digest = hashlib.sha256(archive_path.read_bytes()).hexdigest()
    receipt = destination / "archive_source_sha256.txt"
    if destination.exists() and any(destination.iterdir()):
        if not receipt.is_file() or receipt.read_text().strip() != digest:
            raise FileExistsError("目标目录已有不同内容；请设置新的 ARCHIVE_DESTINATION，避免覆盖修改")
        print("Same archive already extracted; preserving existing project edits.")
    else:
        with zipfile.ZipFile(archive_path) as archive:
            infos = archive.infolist()
            if sum(info.file_size for info in infos) > 2 * 2**30:
                raise ValueError("项目包解压体积超过 2GiB；请勿把基础权重放进代码包")
            for info in infos:
                target = (destination / info.filename).resolve()
                if not target.is_relative_to(destination) or (info.external_attr >> 16) & 0o170000 == 0o120000:
                    raise ValueError("Unsafe archive member")
            destination.mkdir(parents=True, exist_ok=True)
            archive.extractall(destination)
        receipt.write_text(digest + "\n")
    archive_roots = [destination, destination / "cellquant_research_kit"]

candidates = ([Path(PROJECT_ROOT)] if PROJECT_ROOT else []) + archive_roots + [
    Path.cwd(), Path.cwd().parent,
    Path("/content/cellquant_research_kit"),
    Path("/root/autodl-tmp/cellquant_research_kit"),
]
roots = [p.resolve() for p in candidates
         if (p / "pyproject.toml").is_file() and (p / "src/cellquant").is_dir()]
if not roots:
    raise FileNotFoundError("请先解压完整项目包，并设置 PROJECT_ROOT；只上传 notebook 不足以运行。")
PROJECT_ROOT = roots[0]
sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.environ["CELLQUANT_PROJECT_ROOT"] = str(PROJECT_ROOT)

if INSTALL_BASE_DEPENDENCIES:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r",
                           str(PROJECT_ROOT / "requirements/base.txt")])

from cellquant.runtime import (atomic_json, environment_report, resolve_paths,
                              build_manifest, verify_manifest, copy_completed_run)
PATHS = resolve_paths(PROJECT_ROOT, RUN_ROOT, DURABLE_ROOT)
print("Project:", PATHS["project"])
print("Runs:", PATHS["runs"])

Project: /workspace/scratch/506a2af5b817/cellquant_research_kit
Runs: /workspace/scratch/506a2af5b817/cellquant_research_kit/runs


In [4]:
from cellquant.runner import make_job_manifest, check_execution_readiness, execute_manifest
OUT = PATHS["runs"] / "03_full"
OUT.mkdir(parents=True, exist_ok=True)
config_path = PROJECT_ROOT / "configs/full_experiments.json"
full_cfg = json.loads(config_path.read_text()) if config_path.exists() else {"jobs":[], "gpu_hour_budget":8}
plan = make_job_manifest(full_cfg.get("jobs", []), project_root=PROJECT_ROOT,
                         gpu_hour_budget=full_cfg.get("gpu_hour_budget", 8), required_gate="G3",
                         execution_provenance_path=full_cfg.get("execution_provenance_path"))
atomic_json(OUT / "full_manifest.json", plan)
GATE_PATH = PATHS["runs"] / "gates/G3.json"
EVIDENCE_PATH = PATHS["runs"] / "gates/G3_evidence.json"
readiness = check_execution_readiness(plan, gate_path=GATE_PATH, gate_evidence_path=EVIDENCE_PATH)
print("Planned jobs:", len(plan["jobs"]))
print("Hard GPU-hour budget:", plan["gpu_hour_budget"])
print(json.dumps(readiness, ensure_ascii=False, indent=2))

Planned jobs: 32
Hard GPU-hour budget: 240.0
{
  "status": "BLOCKED",
  "missing": [
    "Experiment backend missing callable run_experiment / BACKEND_CONTRACT_VERSION='1.0'",
    "Verified model/tokenizer/data/protocol provenance file missing",
    "Manifest was planned without verified execution provenance",
    "Required gate report is missing"
  ],
  "scope": "Execution prerequisites only; the gate evaluator owns scientific acceptance"
}


## 执行开关

下面使用非阻塞日志轮询，支持中断，不使用可能长时间卡住的 `for line in process.stdout`。子进程超出剩余预算会停止并保存状态。

如果你按“运行全部”，默认开关仍为 False，不会意外开始整个论文矩阵。只有在你已经完成本项目定义的 G3 并决定启动该清单时，才将它改为 True；Codex 不应另造口头审批流程。

In [5]:
RUN_FULL = False
execution = execute_manifest(
    plan, project_root=PROJECT_ROOT, gate_path=GATE_PATH,
    gate_evidence_path=EVIDENCE_PATH, state_dir=OUT / "execution",
    allow_execution=RUN_FULL,
)
print(json.dumps(execution, ensure_ascii=False, indent=2))

{
  "status": "MANIFEST_ONLY",
  "readiness": {
    "status": "BLOCKED",
    "missing": [
      "Experiment backend missing callable run_experiment / BACKEND_CONTRACT_VERSION='1.0'",
      "Verified model/tokenizer/data/protocol provenance file missing",
      "Manifest was planned without verified execution provenance",
      "Required gate report is missing"
    ],
    "scope": "Execution prerequisites only; the gate evaluator owns scientific acceptance"
  },
  "jobs_executed": 0
}


## 已完成的作业不等于论文结论

后端应保存每个任务的 checkpoint/config/data/protocol 哈希、各阶段耗时、GPU/CPU 峰值、真实结果和统计单位。`PROCESS_FINISHED` 只表示程序退出；必须存在合法的 `REAL_MODEL_EXPERIMENT` 工件才能记为 COMPLETE。

缺失任务或种子、不同量化粒度、论文引用结果都不能静默平均到统一表中。跑完后进入 04，并分别生成 G3/G4 的真实证据文件。